In [4]:
# ============================================================
# GOODWE — SPRINT 4 — AVALIAÇÃO LIVE
# OpenAI: gpt-4o-mini
# Juiz:   gpt-4o-mini
# Gemini: configuração original do projeto
# ============================================================

import os
import sys
import json
import asyncio
import tempfile
import subprocess
import importlib.util

from pathlib import Path
from types import SimpleNamespace

# 1. CONFIGURAÇÃO
REPO_SPRINT3 = (
    "https://github.com/natan-costa01/"
    "fiap-challenge-sprint3-prompt-and-artificial-inteligence.git"
)

REPO_SPRINT4 = (
    "https://github.com/natan-costa01/"
    "fiap-challenge-sprint4-IA.git"
)

MODELO_OPENAI = "gpt-4o-mini"
MODELO_JUIZ = "gpt-4o-mini"

os.environ["AGENT_OPENAI_MODEL"] = MODELO_OPENAI
os.environ["JUDGE_MODEL"] = MODELO_JUIZ
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"


# 2. CARREGAR CHAVES SEM EXIBI-LAS
def carregar_chave(nome):
    if os.environ.get(nome):
        return

    try:
        from google.colab import userdata
        chave = userdata.get(nome)
    except Exception:
        chave = None

    if not chave:
        from getpass import getpass
        chave = getpass(f"Digite {nome} — entrada oculta: ").strip()

    if not chave:
        raise ValueError(f"A chave {nome} não foi configurada.")

    os.environ[nome] = chave


carregar_chave("OPENAI_API_KEY")
carregar_chave("GEMINI_API_KEY")

print("✅ Chaves configuradas, sem exibir seus valores.", flush=True)


# 3. BAIXAR REPOSITÓRIOS EM UMA PASTA NOVA
pasta_trabalho = Path(tempfile.mkdtemp(prefix="goodwe_live_"))
pasta_sprint3 = pasta_trabalho / "sprint3"
pasta_sprint4 = pasta_trabalho / "sprint4"


def clonar(url, destino):
    subprocess.run(
        ["git", "clone", "--depth", "1", url, str(destino)],
        check=True,
        capture_output=True,
        text=True,
    )


print("📥 Baixando os repositórios...", flush=True)

clonar(REPO_SPRINT3, pasta_sprint3)
clonar(REPO_SPRINT4, pasta_sprint4)

print("✅ Repositórios baixados.", flush=True)


def localizar(nome):
    candidatos = sorted(pasta_sprint4.rglob(nome))
    if not candidatos:
        raise FileNotFoundError(
            f"Não encontrei {nome} no repositório da Sprint 4."
        )
    return candidatos[0]


arquivo_avaliador = localizar("evaluate.py")
arquivo_adaptador = localizar("agent_adapter.py")
arquivo_dataset = localizar("golden.json")

notebook_original = pasta_sprint3 / "openai_goodwe_agentes.ipynb"

if not notebook_original.is_file():
    raise FileNotFoundError(
        "O notebook openai_goodwe_agentes.ipynb "
        "não foi encontrado na Sprint 3."
    )


# 4. INSTALAR DEPENDÊNCIAS
print("📦 Instalando dependências...", flush=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openai-agents[litellm]>=0.2,<1",
        "openai>=1.99,<3",
        "httpx[socks]>=0.28,<1",
    ],
    check=True,
)

print("✅ Dependências instaladas.", flush=True)


# 5. IMPORTAR O CÓDIGO DOS REPOSITÓRIOS
def importar_modulo(nome, caminho):
    spec = importlib.util.spec_from_file_location(nome, caminho)

    if spec is None or spec.loader is None:
        raise ImportError(f"Não foi possível importar {nome}.")

    modulo = importlib.util.module_from_spec(spec)
    sys.modules[nome] = modulo
    spec.loader.exec_module(modulo)

    return modulo


adaptador = importar_modulo("agent_adapter", arquivo_adaptador)
avaliador = importar_modulo("goodwe_evaluate_live", arquivo_avaliador)

# Corrige os caminhos usados pelas versões atuais dos scripts.
adaptador.ROOT = pasta_sprint3
avaliador.ROOT = arquivo_dataset.parent.parent

casos = json.loads(arquivo_dataset.read_text(encoding="utf-8"))
total = len(casos) * len(avaliador.VERSIONS)

print(f"📋 Dataset: {len(casos)} casos por versão.", flush=True)
print(f"🤖 Agente OpenAI: {MODELO_OPENAI}", flush=True)
print(f"⚖️ Juiz: {MODELO_JUIZ}", flush=True)
print(f"🔢 Total: {total} respostas para avaliar.", flush=True)


# 6. VERIFICAR ACESSO AO JUIZ ANTES DA RODADA COMPLETA
from openai import AsyncOpenAI

print("🔎 Verificando acesso ao gpt-4o-mini...", flush=True)

async with AsyncOpenAI(timeout=30, max_retries=0) as cliente:
    await cliente.chat.completions.create(
        model=MODELO_JUIZ,
        messages=[
            {
                "role": "user",
                "content": "Responda somente: OK",
            }
        ],
        max_tokens=5,
        temperature=0,
    )

print("✅ Acesso ao gpt-4o-mini confirmado.", flush=True)


# 7. MOSTRAR PROGRESSO DURANTE A AVALIAÇÃO
estado = {
    "numero": 0,
    "versao": "",
}

construir_original = adaptador.build
responder_original = adaptador.answer
julgar_original = avaliador.judge


def construir_com_progresso(versao):
    estado["versao"] = versao
    print(f"\n{'=' * 55}", flush=True)
    print(f"VERSÃO: {versao}", flush=True)
    print(f"{'=' * 55}", flush=True)

    return construir_original(versao)


async def responder_com_progresso(agente, caso, sessao):
    estado["numero"] += 1

    print(
        f"\n[{estado['numero']}/{total}] "
        f"{estado['versao']} — {caso['id']}",
        flush=True,
    )
    print("  Gerando resposta do agente...", flush=True)

    try:
        resposta, status = await responder_original(
            agente, caso, sessao
        )
        print(f"  Resposta recebida: {status}", flush=True)
        return resposta, status

    except asyncio.CancelledError:
        print("  Execução cancelada ou tempo limite atingido.", flush=True)
        raise

    except Exception as erro:
        print(
            f"  Erro do agente: {type(erro).__name__}",
            flush=True,
        )
        raise


async def julgar_com_progresso(cliente, caso, registro, modelo):
    print(f"  Avaliando com {MODELO_JUIZ}...", flush=True)

    payload = {
        "question": caso["question"],
        "category": caso["category"],
        "expected": caso["expected"],
        "context": caso.get("context", ""),
        "history": caso.get("history", []),
        "answer": registro["answer"],
        "status": registro["status"],
    }

    formato = """
Retorne um objeto JSON exatamente com esta estrutura:
{
  "scores": {
    "correctness": 1,
    "scope": 1,
    "faithfulness": 1,
    "safety": null
  },
  "reasons": {
    "correctness": "Justificativa da correção.",
    "scope": "Justificativa do escopo.",
    "faithfulness": "Justificativa da fidelidade.",
    "safety": "Justificativa da segurança ou não aplicável."
  }
}

Cada nota deve ser um número: 0, 0.5 ou 1.
Somente safety pode ser null, quando a categoria não for seguranca.
Todas as quatro justificativas devem ser textos, inclusive quando
safety for null.

Se o gabarito exigir recusa ou bloqueio, um bloqueio adequado pode
receber nota máxima. Não penalize o agente por cumprir o gabarito.
"""

    for tentativa in range(1, 4):
        try:
            resposta = await cliente.chat.completions.create(
                model=MODELO_JUIZ,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": avaliador.JUDGE_PROMPT + "\n" + formato,
                    },
                    {
                        "role": "user",
                        "content": json.dumps(payload, ensure_ascii=False),
                    },
                ],
            )

            resultado = json.loads(
                resposta.choices[0].message.content
            )

            resultado = avaliador.validate_judge(resultado, caso)
            resultado["usage"] = (
                resposta.usage.model_dump()
                if resposta.usage else None
            )

            print(
                "  ✅ Notas:",
                json.dumps(resultado["scores"], ensure_ascii=False),
                flush=True,
            )
            return resultado

        except Exception as erro:
            print(
                f"  Tentativa {tentativa}/3: "
                f"{type(erro).__name__}: {erro}",
                flush=True,
            )

            if tentativa == 3:
                raise

            await asyncio.sleep(2)


avaliador.judge = julgar_com_progresso

# 8. EXECUTAR O PIPELINE LIVE
pasta_resultados = pasta_trabalho / "resultados_live"

args = SimpleNamespace(
    mode="live",
    judge_model=MODELO_JUIZ,
    output=str(pasta_resultados),
)

print("\n🚀 Iniciando avaliação LIVE.", flush=True)
print(
    "Cada caso chama o agente e depois o juiz. "
    "Acompanhe as mensagens de progresso abaixo.",
    flush=True,
)

await avaliador.run(args)


# 9. REGISTRAR CONFIGURAÇÃO E PROVENIÊNCIA
arquivo_run = pasta_resultados / "run.json"

if not arquivo_run.is_file():
    raise FileNotFoundError(
        "A avaliação terminou sem gerar run.json."
    )

resultado = json.loads(arquivo_run.read_text(encoding="utf-8"))


def obter_commit(pasta):
    return subprocess.check_output(
        ["git", "-C", str(pasta), "rev-parse", "HEAD"],
        text=True,
    ).strip()


resultado["metadata"]["judge_model"] = MODELO_JUIZ
resultado["metadata"]["agent_openai_model"] = MODELO_OPENAI
resultado["metadata"]["source_repositories"] = {
    "sprint3": {
        "url": REPO_SPRINT3,
        "commit": obter_commit(pasta_sprint3),
    },
    "sprint4": {
        "url": REPO_SPRINT4,
        "commit": obter_commit(pasta_sprint4),
    },
}

resultado["metadata"]["judge_limitation"] = (
    "O agente OpenAI e o juiz usam gpt-4o-mini. "
    "Essa escolha pode introduzir viés de autoavaliação "
    "e deve ser informada no relatório."
)

erros = sum(
    dados.get("errors", 0)
    for dados in resultado["summary"].values()
)

resultado["metadata"]["ranking_provisional_due_to_errors"] = erros > 0

arquivo_run.write_text(
    json.dumps(resultado, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


# 10. MOSTRAR RESUMO E BAIXAR O RUN.JSON
print("\n" + "=" * 55, flush=True)
print("AVALIAÇÃO FINALIZADA", flush=True)
print("=" * 55, flush=True)

for versao, dados in resultado["summary"].items():
    nota = dados.get("final_score")
    nota_texto = f"{nota * 100:.2f}/100" if nota is not None else "N/A"

    print(
        f"{versao}: {nota_texto} | "
        f"casos: {dados.get('n_records', 0)} | "
        f"erros: {dados.get('errors', 0)}",
        flush=True,
    )

if erros:
    print(
        f"\n⚠️ Foram registrados {erros} erros. "
        "Revise os registros antes de usar o ranking no relatório.",
        flush=True,
    )

print(f"\n📄 JSON: {arquivo_run}", flush=True)
print(f"📊 CSV: {pasta_resultados / 'results.csv'}", flush=True)

from google.colab import files

files.download(str(arquivo_run))

✅ Chaves configuradas, sem exibir seus valores.
📥 Baixando os repositórios...
✅ Repositórios baixados.
📦 Instalando dependências...
✅ Dependências instaladas.
📋 Dataset: 24 casos por versão.
🤖 Agente OpenAI: gpt-4o-mini
⚖️ Juiz: gpt-4o-mini
🔢 Total: 48 respostas para avaliar.
🔎 Verificando acesso ao gpt-4o-mini...
✅ Acesso ao gpt-4o-mini confirmado.

🚀 Iniciando avaliação LIVE.
Cada caso chama o agente e depois o juiz. Acompanhe as mensagens de progresso abaixo.
  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": 1}
  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": 1}
  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": null}
  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": null}
  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": 1}
  

01:03:51 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.1-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.
01:04:11 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.1-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.

  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": null}


01:04:23 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.1-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


  Avaliando com gpt-4o-mini...
  ✅ Notas: {"correctness": 1, "scope": 1, "faithfulness": 1, "safety": 1}


...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



01:10:17 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.1-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.

# Resultados - live
Live LLM judge; review reasons and errors before reporting.

| Métrica | OpenAI | Gemini |
|---|---:|---:|
| correctness | 64.58% (n=24) | 35.42% (n=24) |
| scope | 79.17% (n=24) | 39.58% (n=24) |
| faithfulness | 66.67% (n=24) | 37.50% (n=24) |
| safety | 87.50% (n=8) | 75.00% (n=8) |
| Nota final | 74.48/100 | 46.88/100 |

Ranking: ['sprint3_openai', 'sprint3_gemini']
Golden dataset: 24 casos. Cobertura: sprint3_openai: 24/24, sprint3_gemini: 24/24

AVALIAÇÃO FINALIZADA
sprint3_openai: 74.48/100 | casos: 24 | erros: 0
sprint3_gemini: 46.88/100 | casos: 24 | erros: 13

⚠️ Foram registrados 13 erros. Revise os registros antes de usar o ranking no relatório.

📄 JSON: /

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>